# **Introduction to Fourier transform in two dimensions**
### **Author: Dr. Rahul Remanan**

In [ ]:
import os, math, numpy as np, scipy as sp, \
       matplotlib.image as mpimg,          \
       matplotlib.pyplot as plt

from scipy.stats import norm

# Handling complex numbers in NumPy:

Every non-zero complex number ${z = x+ iy}$ can be written in the form: ${re}^{iθ}$, where the value: ${r = |z| = {\sqrt{{x^{2}+y^{2}}}}}$ is referred to as the **magnitude** of $z$, and $θ$ as the **phase, angle, or argument** of $z$.

## Getting the magnitude of a complex number array:

In [ ]:
print(
    '\n',
    'Magnitude: ', np.abs([1.0, 1.0j, 1+1j, -1-1j])
)

## Getting the phase of a complex number array:

In [ ]:
print(
    '\n',
    'In radians: ', np.angle([1.0, 1.0j, 1+1j, -1-1j]),
    '\n',
    'In degrees: ', np.angle([1.0, 1.0j, 1+1j, -1-1j], deg=True),
    '\n',
    np.angle([0., -0., complex(0., -0.), complex(-0., -0.)])   # convention
)

## Getting the sign of a complex number array:

The [sign function](https://numpy.org/doc/stable/reference/generated/numpy.sign.html) is defined as: $S(x + iy) = \frac{(x + iy)}{\sqrt{x^2+y^2}}$

In [ ]:
np.sign([1.0, 1.0j, 1+1j, -1-1j])

# Discrete Fourier transform (DFT) and inverse discrete Fourier transform (iDFT):

In [ ]:
a = np.array(
        [
            [0.0, 0.5, 1.0],
            [1.0, 2.0, 3.0],
            [5.0, 7.0, 8.0],
            [4.0, 9.0, 6.0]
        ]
)

In [ ]:
N = np.prod(a.shape)

## **Discrete Fourier transform:**

Discrete Fourier transform of a two dimensional $M{\times}N$ matrix, decomposes each value of the matrix: $r_{n}{\text{, where: }}r{\in}{ℝ}$,  into the form: ${c_{n}} = {a_{n}}({cos}({\theta}_{n}) + {i}{\cdot}{sin}({\theta}_{n}))$, through the application of: 
* [Euler's formula](https://en.wikipedia.org/wiki/Euler's_formula): ${e^{i{\cdot}{x}}={{\cos{x}}+{i}{\cdot}{\sin{x}}}}$
* [Improper Reimann integral](https://en.wikipedia.org/wiki/Improper_integral).

${F(a,b)={{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}f(k,l){e^{{-i}{2}{\pi}{({\frac{ka}{N}}+{\frac{lb}{M}})}}}}$

In [ ]:
def fourier_transform_exp(d):
    e_ft = np.exp(-1 * (np.array([0 + 1j]) * 2 * math.pi * d))
    return e_ft.astype(np.complex128)

def compute_fourier_transform(inp_arr):
    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'
    N, M = inp_arr.shape[0], inp_arr.shape[1]
    a_ft = np.zeros_like(inp_arr).astype(np.complex128)
    for i in range(a_ft.shape[0]):
        for j in range(a_ft.shape[1]):
            ft = 0
            for k in range(N):
                for l in range(M):
                    d   = ((k * i) / N) + ((l * j) / M)
                    ft += inp_arr[k, l] * fourier_transform_exp(d)
            a_ft[i, j] = ft[0]
    return a_ft

In [ ]:
a_ft = compute_fourier_transform(a)

In [ ]:
print(a.shape, a_ft.shape)

In [ ]:
print(f'\nFourier transform output: \n{a_ft}')

## **Inverse discrete Fourier transform:**

Inverse discrete Fourier transform of a two dimensional $M{\times}N$ matrix, recomposes each value of the matrix: ${{c_{n}}{\text{, where: c}}{\in}{ℂ}}$, into the corresponding value: ${r_{n}}{\text{, where: r}}{{\in}ℝ}$, of another $M{\times}N$ matrix; which was used to derive the discrete Fourier transformed complex matrix. 

The inverse discrete Fourier transform function maps: ${c_{n}{\rightarrow}r_{n}}$, through the application of:
* [Euler's formula](https://en.wikipedia.org/wiki/Euler's_formula): ${e^{i{\cdot}{x}}={{\cos{x}}+{i}{\cdot}{\sin{x}}}}$
* [Improper Reimann integral](https://en.wikipedia.org/wiki/Improper_integral).

${f(x,y)={{\frac{1}{N{\cdot}M}}{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}F(k,l){e^{{i}{2}{\pi}{({\frac{kx}{N}}+{\frac{ly}{M}})}}}}$

In [ ]:
def inverse_fourier_transform_exp(d):
    e_ift = np.exp(np.array([0 + 1j]) * 2 * math.pi * d)
    return e_ift.astype(np.complex128)

def compute_inverse_fourier_transform(inp_arr):
    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'
    N, M = inp_arr.shape[0], inp_arr.shape[1]
    a_ift = np.zeros_like(inp_arr).astype(np.complex128)
    for i in range(a_ift.shape[0]):
        for j in range(a_ift.shape[1]):
            ift = 0
            for k in range(N):
                for l in range(M):
                    d    = ((k * i) / N) + ((l * j) / M)
                    ift += inp_arr[k, l].astype(np.complex128) * inverse_fourier_transform_exp(d)
            a_ift[i, j] = ift[0] # np.array([ift]).astype(np.complex128)
    return a_ift / np.prod(inp_arr.shape)

In [ ]:
a_ift = compute_inverse_fourier_transform(a_ft)

In [ ]:
print(f'\nInverse Fourier transform output: \n{a_ift}')

## **Error in estimating the original input matrix using inverse discrete Fourier transform:**

In [ ]:
print(f'\nError between original 2D matrix input and inverse Fourier transform matrix: \n{a_ift - a}')

# Separable functions of discrete Fourier transform:

$
\text{The discrete Fourier transform function: }
$

$
{F(a,b)={{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}f(k,l){e^{{-i}{2}{\pi}{({\frac{ka}{N}}+{\frac{lb}{M}})}}}}
\text{, can be expressed as two separable functions:}$

${P(a,l)={{\sum\limits_{k=0}^{N-1}}}f(k,l){e^{{-i}{2}{\pi}{({\frac{ka}{N}})}}}}$

${F(a,b)={{\sum\limits_{l=0}^{M-1}}}P(a,l){e^{{-i}{2}{\pi}{({\frac{lb}{M}})}}}}$

In [ ]:
def p_fourier_transform_sep(inp_arr, i, l):
    N = inp_arr.shape[0]
    p_ft = 0
    for k in range(N):
        d = ((k * i) / N)
        p_ft += inp_arr[k, l] * fourier_transform_exp(d)
    return p_ft

def fourier_transform_sep(inp_arr, i, j):
    M = inp_arr.shape[1]
    ft = 0
    for l in range(M):
        d = ((l * j) / M)
        ft += p_fourier_transform_sep(inp_arr, i, l) * fourier_transform_exp(d)
    return ft

def compute_fourier_transform_sep(inp_arr):
    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'
    a_ft_sep = np.zeros_like(inp_arr).astype(np.complex128)
    N, M = a_ft_sep.shape[0], a_ft_sep.shape[1]
    for i in range(N):
        for j in range(M):
            a_ft_sep[i, j] = fourier_transform_sep(inp_arr, i, j)[0]
    return a_ft_sep

In [ ]:
a_ft_sep = compute_fourier_transform_sep(a)

In [ ]:
print(f'\nOutput of the Fourier transform using separable functions: \n{a_ft_sep}')

## **Difference in results using two methods of evaluating discrete Fourier transform:**

In [ ]:
a_ft - a_ft_sep

# Separable functions of inverse discrete Fourier transform:

$
\text{The inverse discrete Fourier transform function: }
$

$
{f(x,y)={{\frac{1}{N{\cdot}M}}{\sum\limits_{k=0}^{N-1}}{\sum\limits_{l=0}^{M-1}}}F(k,l){e^{{i}{2}{\pi}{({\frac{kx}{N}}+{\frac{ly}{M}})}}}}
\text{, can be expressed as two separable functions:}$

${p(x,l)={{\frac{1}{N}}{\sum\limits_{k=0}^{N-1}}}F(k,l){e^{{i}{2}{\pi}{({\frac{kx}{N}})}}}}$

${f(x,y)={{\frac{1}{M}}{{\sum\limits_{l=0}^{M-1}}}}p(x,l){e^{{i}{2}{\pi}{({\frac{ly}{M}})}}}}$

In [ ]:
def p_inverse_fourier_transform_sep(inp_arr, i, l):
    N = inp_arr.shape[0]
    p_ift = 0
    for k in range(N):
        d      = ((k * i) / N)
        p_ift += inp_arr[k, l] * inverse_fourier_transform_exp(d)
    return p_ift / N

def inverse_fourier_transform_sep(inp_arr, i, j):
    M = inp_arr.shape[1]
    ift = 0
    for l in range(M):
        d    = ((l * j) / M)
        ift += p_inverse_fourier_transform_sep(inp_arr, i, l) * inverse_fourier_transform_exp(d)
    return ift / M

def compute_inverse_fourier_transform_sep(inp_arr):
    assert len(inp_arr.shape) == 2, f'Expected a two dimensional input array, instead received an array of: {inp_arr.shape}  dimensions ...'
    N, M = inp_arr.shape[0], inp_arr.shape[1]
    a_ift_sep = np.zeros_like(inp_arr).astype(np.complex128)
    for i in range(N):
        for j in range(M):
            a_ift_sep[i, j] = inverse_fourier_transform_sep(inp_arr, i, j)[0]
    return a_ift_sep

In [ ]:
a_ift_sep = compute_inverse_fourier_transform_sep(a_ft_sep)

In [ ]:
print(f'\nOutput of the inverse Fourier transform using separable functions: \n{a_ift_sep}')

## **Error in estimating the original input matrix using inverse discrete Fourier transform:**

In [ ]:
a_ift_sep - a

# Fourier transform of a discrete signal using NumPy:

In [ ]:
# Source - https://stackoverflow.com/q/52350774
# Posted by Luke Polson
# Retrieved 2026-04-03, License - CC BY-SA 4.0

# Sample spacing
dx = 1.0 / 1000.0

# Points
x1 = -5
x2 = 5

x = np.arange(x1, x2, dx)

def light_intensity():
    return 10 * sp.stats.norm.pdf(x, 0, 1)+ 0.1 * np.random.randn(x.size)

fig, ax = plt.subplots()
ax.plot(x,light_intensity())

In [ ]:
# Source - https://stackoverflow.com/a/52351218
# Posted by Mateen Ulhaq, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-03, License - CC BY-SA 4.0
# Modified by Dr. Rahul Remanan after retrieval

def norm_fft(y, T, max_freq=None):
    N = y.shape[0]
    Nf = N // 2 if max_freq is None else int(max_freq * T)
    xf = np.linspace(0.0, 0.5 * N / T, N // 2)
    yf = 2.0 / N * np.fft.fft(y)
    return xf[:Nf], yf[:Nf]

def generate_signal(x, signal_gain=10.0, noise_gain=0.0):
    signal = norm.pdf(x, 0, 1)
    noise = np.random.randn(x.size)
    return signal_gain * signal + noise_gain * noise

# Signal parameters
x1 = 0.0
x2 = 5.0
N = 10000
T = x2 - x1

# Generate signal data
x = np.linspace(x1, x2, N)
y = generate_signal(x)

# Apply FFT
xf, yf = norm_fft(y, T, T / np.pi)

# Plot
fig, ax = plt.subplots(2)
ax[0].plot(x, y)
ax[1].plot(xf, np.abs(yf))
plt.show()

In [ ]:
# Source - https://stackoverflow.com/a/52351218
# Posted by Mateen Ulhaq, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-03, License - CC BY-SA 4.0
# Modified by Dr. Rahul Remanan after retrieval

def norm_sym_fft(y, T, max_freq=None):
    N = y.shape[0]
    b = N if max_freq is None else int(max_freq * T + N // 2)
    a = N - b
    xf = np.linspace(-0.5 * N / T, 0.5 * N / T, N)
    yf = 2.0 / N * np.fft.fftshift(np.fft.fft(y))
    return xf[a:b], yf[a:b]

def generate_signal(x, signal_gain=10.0, noise_gain=0.0):
    signal = norm.pdf(x, 0, 1)
    noise = np.random.randn(x.size)
    return signal_gain * signal + noise_gain * noise

# Signal parameters
x1 = -10.0
x2 = 10.0
N = 10000
T = x2 - x1

# Generate signal data
x = np.linspace(x1, x2, N)
y = generate_signal(x)

# Apply FFT
xf, yf = norm_sym_fft(y, T, 4 / np.pi)

# Plot
fig, ax = plt.subplots(2)
ax[0].plot(x, y)
ax[1].plot(xf, np.abs(yf))
plt.show()

In [ ]:
# Source - https://stackoverflow.com/a/52351212
# Posted by Mahdi
# Retrieved 2026-04-03, License - CC BY-SA 4.0
# Modified by Dr. Rahul Remanan after retrieval

# Sample spacing
dx = 1.0 / 1000.0

# Points
x1 = -5
x2 = 5

x = np.arange(x1, x2, dx)

xf = np.arange(x1, x2, dx)
yf = np.fft.fft(light_intensity())
yfft = np.fft.fftshift(np.abs(yf))
fig,ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].plot(xf, light_intensity())
ax[1].plot(xf, yfft)
ax[1].set_xlim(-0.05, 0.05)
plt.show()

# [Fast Fourier transform of a grey scale image](https://homepages.inf.ed.ac.uk/rbf/HIPR2/fourier.htm):

In [ ]:
img_file = '../media/cln1.gif'
if not os.path.exists(img_file):
    !curl https://homepages.inf.ed.ac.uk/rbf/HIPR2/images/cln1.gif >> {img_file}

In [ ]:
img = mpimg.imread(img_file)

In [ ]:
imgplot = plt.imshow(
    np.expand_dims(
        img[ : , : , 0], axis=-1
    )
)

In [ ]:
img_fft = np.fft.fft(

    img[ : , : , 0]

)

In [ ]:
print(f'\nOutput of the fast Fourier transformed grey scale image: \n{img_fft}')

In [ ]:
np.max(np.abs(img_fft)), np.min(np.abs(img_fft))

In [ ]:
def min_max_scaler(img):
    if np.max(img) - np.min(img) != 0:
        return (img - np.min(img)) / (np.max(img) - np.min(img))
    return img

def log_transform(img):
    return (1 / (np.log(1 + np.max(img)))) * np.log(1 + img)

In [ ]:
img_fft_mag_plot = plt.imshow(
    np.expand_dims(
        log_transform(
            (
                np.abs(
                    img_fft
                )
            )
        ), axis=-1
    )
)

In [ ]:
img_fft_phase_plot = plt.imshow(
    np.expand_dims(
        log_transform(
            min_max_scaler(
                np.angle(
                    img_fft
                )
            )
        ), axis=-1
    )
)

# Inverse fast Fourier transform:

In [ ]:
img_ifft = np.fft.ifft(

    img_fft

)

In [ ]:
img_fft_mag_plot = plt.imshow(
    np.abs(
        np.expand_dims(
            img_ifft, axis=-1
        )
    )
)

## Inverse Fourier transform without the phase information:

In [ ]:
img_ifft_mag_plot = plt.imshow(
    np.expand_dims(
        log_transform(
            np.abs(
                np.fft.ifft(
                    np.abs(
                        img_fft
                    )
                )
            )
        ), axis=-1
    )
)

## Inverse Fourier transform using magnitude and phase as separable components:

In [ ]:
img_ifft_mag_plot = plt.imshow(
    np.expand_dims(
        min_max_scaler(
            np.abs(
                np.fft.ifft(
                    np.abs(
                        img_fft
                    ) * np.sign(
                            img_fft
                        )
                )
            )
        ), axis=-1
    )
)